# BOPP coordinates: unit ⇒ number type

Proposal: model extents the way [timetoalign](https://timetoalign.github.io) does. A **coordinate** is a
position on a timeline, i.e. a *value* measured in a *unit*. Every unit is coupled to exactly one number type:

| unit class | units | number type | buffer |
|---|---|---|---|
| discrete | `ticks`, `samples`, `frames`, `pixels` | `int` | `Int64Buffer` |
| continuous symbolic | `quarters`, `whole_note` | `fraction` | `FractionBuffer` (`numerator[]`, `denominator[]`) |
| continuous | `seconds`, `milliseconds`, `minutes`, `floating_measures`, `number`, `meters`, … | `float` | `Float64Buffer` |

`schemas/v1/coordinate.json` is a `oneOf` over the units, discriminated by `unit`. `times` carries one coordinate,
`time_interval` two (`start`, `duration`). Since the unit tag fully determines the buffer type, readers never guess
precision from the data, and "which unit is this extent in?" no longer needs a dedicated `extent_type` per unit
(the previous `quarter_interval` is now simply `time_interval` + `unit: quarters`).

In [1]:
import json
from pathlib import Path

import msgspec

from bopp.io import decode_arrow, encode_arrow, to_csv, load_bopp_csv
from bopp.models.v1.annotation import Annotation
from bopp.models.v1 import coordinate
from bopp.util import to_dataframe

schema = json.loads(Path('schemas/v1/coordinate.json').read_text(encoding='utf-8'))
print(schema['description'])
print()
for name, variant in schema['$defs'].items():
    print(f"{variant['properties']['unit']['const']:>18} -> {variant['properties']['values']['$ref'].split('/')[-1]}")

A columnar array of positions (or elapsed extents) on a timeline, following the timetoalign model: a coordinate is a value measured in a unit, and every unit is coupled to exactly one number type. Discrete units (ticks, samples, frames, pixels) are int; continuous symbolic units (quarters, whole_note) are exact fractions; all other continuous units are float. The `unit` tag therefore fully determines the buffer type, so consumers never guess precision from the data.

            number -> Float64Buffer
 floating_measures -> Float64Buffer
        whole_note -> FractionBuffer
          quarters -> FractionBuffer
             ticks -> Int64Buffer
      milliseconds -> Float64Buffer
           seconds -> Float64Buffer
           minutes -> Float64Buffer
           samples -> Int64Buffer
            frames -> Int64Buffer
            pixels -> Int64Buffer
            meters -> Float64Buffer
       centimeters -> Float64Buffer
       millimeters -> Float64Buffer
            inches -> Float64B

## The generated msgspec model

One tagged struct per unit; the union is what `times.time`, `time_interval.start` and `time_interval.duration` are typed with.

In [2]:
print(Path('bopp/models/v1/extent/time_interval.py').read_text(encoding='utf-8'))
print(coordinate.QuartersCoordinate.__annotations__)

# @generated by datamodel-codegen:
#   filename:  extent/time_interval.json

from __future__ import annotations

from typing import Annotated

from msgspec import Meta, Struct

from .. import coordinate


class TimeIntervalExtent(Struct, tag_field="extent_type", tag="time_interval"):
    """
    Left-inclusive, right-exclusive intervals [start, start + duration), expressed as two coordinates that must share the same unit.
    """

    start: Annotated[
        coordinate.Coordinate, Meta(description="N interval start positions.")
    ]
    duration: Annotated[
        coordinate.Coordinate,
        Meta(description="N interval durations (elapsed extents), same unit as start."),
    ]

{'values': "Annotated[core.FractionBuffer, Meta(description='Coordinate values in quarters, stored as fraction.')]"}


## Physical time: `seconds` → float64

In [3]:
def annotation(extent, payload_values=('C:maj', 'G:maj', 'C:maj')):
    raw = {
        'media_id': 'demo:coordinates',
        'bopp_version': '1.0.0',
        'extent': extent,
        'payload': {'payload_type': 'chord', 'values': list(payload_values)},
    }
    return msgspec.convert(raw, type=Annotation, dec_hook=decode_arrow)

seconds = annotation({
    'extent_type': 'time_interval',
    'start': {'unit': 'seconds', 'values': [0.0, 1.5, 3.25]},
    'duration': {'unit': 'seconds', 'values': [1.5, 1.75, 2.0]},
})
print(type(seconds.extent.start).__name__, seconds.extent.start.values.type)
print(msgspec.json.encode(seconds, enc_hook=encode_arrow).decode())

SecondsCoordinate double
{"object_type":"annotation","media_id":"demo:coordinates","payload":{"payload_type":"chord","values":["C:maj","G:maj","C:maj"]},"bopp_version":"1.0.0","extent":{"extent_type":"time_interval","start":{"unit":"seconds","values":[0.0,1.5,3.25]},"duration":{"unit":"seconds","values":[1.5,1.75,2.0]}}}


## Discrete time: `samples` → int64

A non-integral sample position is not a rounding candidate but a modelling error, so it is rejected.

In [4]:
samples = annotation({'extent_type': 'timestamps', 'time': {'unit': 'samples', 'values': [0, 66150, 143325]}})
print(type(samples.extent.time).__name__, samples.extent.time.values.type)

try:
    annotation({'extent_type': 'timestamps', 'time': {'unit': 'samples', 'values': [0, 66150.5, 143325]}})
except Exception as error:
    print('rejected:', error)

SamplesCoordinate int64
rejected: SamplesCoordinate.values must be int64: Float value 66150.500000 was truncated converting to int64


## Symbolic time: `quarters` → exact fractions

Notated rhythm is rational by construction: a triplet eighth is 1/3 of a quarter, not 0.333…. The buffer is a struct-of-arrays (`numerator[]`, `denominator[]`), which stays columnar and maps onto an Arrow struct.

In [5]:
quarters = annotation({
    'extent_type': 'time_interval',
    'start': {'unit': 'quarters', 'values': {'numerator': [0, 1, 4], 'denominator': [1, 3, 3]}},
    'duration': {'unit': 'quarters', 'values': {'numerator': [1, 1, 2], 'denominator': [3, 1, 1]}},
})
print(msgspec.json.encode(quarters.extent, enc_hook=encode_arrow).decode())
to_dataframe(quarters)

{"extent_type":"time_interval","start":{"unit":"quarters","values":{"numerator":[0,1,4],"denominator":[1,3,3]}},"duration":{"unit":"quarters","values":{"numerator":[1,1,2],"denominator":[3,1,1]}}}


,extent:time_interval:start:quarters,extent:time_interval:duration:quarters,payload:chord:values
0,0,1/3,C:maj
1,1/3,1,G:maj
2,4/3,2,C:maj


The CSV header carries the unit as a fourth segment (`extent:time_interval:start:quarters`), which is all a reader needs to restore the exact number type.

In [6]:
to_csv(quarters, 'coordinates_demo.bopp.csv')
print(Path('coordinates_demo.bopp.csv').read_text(encoding='utf-8'))
restored = load_bopp_csv('coordinates_demo.bopp.csv')
assert msgspec.msgpack.encode(restored, enc_hook=encode_arrow) == msgspec.msgpack.encode(quarters, enc_hook=encode_arrow)
print('CSV round trip is lossless; start numerators:', restored.extent.start.values.numerator.to_pylist())

# ---
# object_type: annotation
# media_id: demo:coordinates
# bopp_version: 1.0.0
# ---
extent:time_interval:start:quarters,extent:time_interval:duration:quarters,payload:chord:values
0,1/3,C:maj
1/3,1,G:maj
4/3,2,C:maj



CSV round trip is lossless; start numerators: [0, 1, 4]


## Guard rails

The schema cannot express that `start` and `duration` share a unit; the `Annotation` model checks it at construction (next to the existing parallel-length check).

In [7]:
try:
    annotation({
        'extent_type': 'time_interval',
        'start': {'unit': 'quarters', 'values': {'numerator': [0, 1, 2], 'denominator': [1, 1, 1]}},
        'duration': {'unit': 'seconds', 'values': [1.0, 1.0, 1.0]},
    })
except Exception as error:
    print('rejected:', error)

try:
    annotation({'extent_type': 'timestamps', 'time': {'unit': 'seconds', 'values': [0.0, 1.0]}})
except Exception as error:
    print('rejected:', error)

rejected: Unit mismatch: extent mixes coordinates in ['quarters', 'seconds'].
rejected: Length mismatch: Found multiple array lengths {2, 3} across Extent, Payload, and Confidence facets.


## Size: msgpack keeps fractions cheap

1000 quarter positions as exact fractions vs. the same positions as float32 seconds.

In [8]:
from fractions import Fraction
positions = [Fraction(i, 3) for i in range(1000)]
exact = annotation({
    'extent_type': 'timestamps',
    'time': {'unit': 'quarters', 'values': {'numerator': [p.numerator for p in positions], 'denominator': [p.denominator for p in positions]}},
}, payload_values=['C:maj'] * 1000)
approx = annotation({'extent_type': 'timestamps', 'time': {'unit': 'seconds', 'values': [float(p) for p in positions]}}, payload_values=['C:maj'] * 1000)
for label, obj in [('quarters (fraction)', exact), ('seconds (float64)', approx)]:
    print(f"{label:>20}: msgpack {len(msgspec.msgpack.encode(obj, enc_hook=encode_arrow)):>6} B   json {len(msgspec.json.encode(obj, enc_hook=encode_arrow)):>6} B")

 quarters (fraction): msgpack   9555 B   json  14051 B
   seconds (float64): msgpack  15167 B   json  22512 B
